# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrahmanshaheen1/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Primary task: Ranking / scoring

My lane is primarily a ranking and scoring task. The system should give each webpage a priority score and then order the pages from highest to lowest priority. The highest-ranked pages would be reviewed first by the content team.

A classification model may be used to estimate the probability that a page is declining, but the business does not only need a yes-or-no answer. The team has limited time and needs to know which pages should be checked first. Therefore, the final task is ranking, while classification may be one method used to create the ranking score.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Provisional target: whether a page is declining

The provisional target is a binary column called `is_declining`.

- `1` means the page is labelled as declining.
- `0` means the page is not labelled as declining.

This proxy comes from the observed `trend_direction` column in the starter dataset:

`is_declining = 1` when `trend_direction` is `"down"`.

This is a temporary proxy for the current assignment, not a perfect future target. It tells us which pages are currently labelled as declining, but it does not prove why they declined or guarantee that refreshing them will improve performance.

Because `trend_direction` and `trend_pct` reveal how this label was created, they must not be used as model input features. Using them would give the model part of the answer and cause data leakage.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric: Precision@50

The main success metric will be Precision@50. It measures how many of the top 50 pages recommended for review are actually labelled as declining.

For example, a Precision@50 of 0.70 means that 35 of the top 50 recommended pages are declining.

A useful result should perform better than the fixed-rule baseline from the starter notebook, which achieved about 0.68 Precision@50. Therefore, a provisional goal is to achieve more than 0.68 on unseen data while keeping the recommendations understandable and useful for the content team.

This metric fits the decision because the team has limited time and mainly cares about whether the pages at the top of the queue are worth reviewing.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

data_url = (
    "https://raw.githubusercontent.com/"
    "Abdelrahmanshaheen1/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

# Provisional target:
# 1 = declining page, 0 = not declining
df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Page-level columns relevant to Lane 2
candidate_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "word_count",
    "is_declining"
]

# Keep only columns that exist in the starter dataset
available_columns = [
    column for column in candidate_columns
    if column in df.columns
]

lane_df = df[available_columns].copy()

print("Unit of analysis: one row represents one webpage.")
print(f"Lane dataframe shape: {lane_df.shape[0]:,} rows × {lane_df.shape[1]} columns")
print("\nExample page-level rows:")

lane_df.head()

Unit of analysis: one row represents one webpage.
Lane dataframe shape: 30,000 rows × 8 columns

Example page-level rows:


,search_volume,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,word_count,is_declining
0,10.0,3803,29,0.76,10.6,187,3221.0,1
1,90.0,15320,7,0.05,20.3,445,2481.0,1
2,0.0,12581,11,0.09,36.5,141,3515.0,1
3,10.0,11751,58,0.49,6.2,463,NaN,0
4,0.0,19140,24,0.13,44.0,263,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML may be more useful than one fixed rule

A fixed rule might say that every old page with many impressions should be reviewed. However, page performance depends on several signals that may interact in different ways, such as impressions, clicks, CTR, average position, content age, word count, and freshness.

For example, an old page may still be performing well, while a newer page may already be declining. A page with many impressions may be important, but its average position and CTR may change how urgently it should be reviewed. One simple if-statement may miss these combinations.

A machine-learning model may learn patterns across several signals and produce a priority score for each page. The score can then be used to rank pages for human review. The model should still be compared with a simple fixed-rule baseline and tested on unseen data. Its output would support a content decision, not automatically decide what should happen to a page.

## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [✔ ] My claims use careful words: observed, measured, directional, decision-support
- [✔ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.